# Matrix Inverse using Gaussian Elimination

## Overview
To find the inverse of a matrix $A$, we solve the equation:
$$A \cdot X = I$$

where $X = A^{-1}$ is the inverse and $I$ is the identity matrix.

### Key Insight
We can find $X$ by solving **one system for each column** of the inverse:
$$A \cdot \mathbf{x}_i = \mathbf{e}_i$$

where $\mathbf{e}_i$ is the $i$-th **standard basis vector** (column $i$ of the identity matrix).

### Example
- **2×2 matrix**: Solve 2 systems (each 2×2) = **2 equations total**
- **3×3 matrix**: Solve 3 systems (each 3×3) = **3 equations total**

We will use the `gaussian_elimination` function from **math_005_gaussian_elimination** notebook.

## Step 1: Import Libraries and Load Gaussian Elimination Function

In [22]:
import numpy as np
import matplotlib.pyplot as plt

# Gaussian elimination function (from math_005_gaussian_elimination)
def gaussian_elimination(A, b):
    """Solve Ax = b using Gaussian elimination (forward and backward elimination).
    
    Parameters:
    -----------
    A : ndarray
        Coefficient matrix (n x n)
    b : ndarray
        Constant vector (n x 1) or (n,)
    
    Returns:
    --------
    x : ndarray
        Solution vector (n,)
    """
    m = np.hstack([A.astype(float).copy(), b.astype(float).reshape(-1, 1)])
    rows, cols = m.shape
    n = cols - 1

    # Forward elimination: create zeros below pivots
    for pivot in range(n):
        # Partial pivoting for numerical stability
        max_row = pivot + np.argmax(np.abs(m[pivot:, pivot]))
        if abs(m[max_row, pivot]) < 1e-12:
            raise ValueError('Matrix is singular or nearly singular.')
        if max_row != pivot:
            m[[pivot, max_row]] = m[[max_row, pivot]]

        for r in range(pivot + 1, rows):
            factor = m[r, pivot] / m[pivot, pivot]
            m[r] = m[r] - factor * m[pivot]

    # Normalize pivots to 1
    for pivot in range(n):
        m[pivot] = m[pivot] / m[pivot, pivot]

    # Backward elimination: create zeros above pivots
    for pivot in range(n - 1, -1, -1):
        for r in range(pivot - 1, -1, -1):
            factor = m[r, pivot]
            m[r] = m[r] - factor * m[pivot]

    # Final solution vector from the last column
    return m[:, -1]

print('✓ Gaussian elimination function loaded')

✓ Gaussian elimination function loaded


## Step 2: Function to Compute Matrix Inverse using Gaussian Elimination

We create a function that:
1. For each column $i$ of the inverse, solve $A \cdot \mathbf{x}_i = \mathbf{e}_i$
2. Collect all solutions as columns of the inverse matrix

In [23]:
def compute_matrix_inverse_gaussian_elimination(A):
    """Compute matrix inverse A^-1 using Gaussian elimination.
    
    Algorithm:
    -----------
    To find A^-1 such that A * X = I, solve n systems:
        A * x_i = e_i    (for i = 1, 2, ..., n)
    
    where e_i is the i-th standard basis vector (column i of identity matrix).
    
    The solution vectors x_i become the columns of A^-1.
    
    Parameters:
    -----------
    A : ndarray
        Square matrix (n x n) to invert
    
    Returns:
    --------
    A_inv : ndarray
        Inverse matrix (n x n)
    """
    n = A.shape[0]
    inv_A = np.zeros((n, n))
    
    # For each column of the inverse matrix
    for col in range(n):
        # Create the standard basis vector e_i
        e_i = np.zeros(n)
        e_i[col] = 1.0
        
        # Solve A @ x_i = e_i using Gaussian elimination
        x_i = gaussian_elimination(A, e_i)
        
        # Store the solution as column col of A_inv
        inv_A[:, col] = x_i
    
    return inv_A

print('✓ Matrix inverse function defined')

✓ Matrix inverse function defined


## Step 3: Example 1 - 2×2 Matrix Inverse

For a 2×2 matrix:
$$A = \begin{bmatrix} 4 & 7 \\ 2 & 6 \end{bmatrix}$$

We solve **2 systems**:
1. $A \cdot \mathbf{x}_1 = \mathbf{e}_1 = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$
2. $A \cdot \mathbf{x}_2 = \mathbf{e}_2 = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$

In [24]:
print('=' * 70)
print('EXAMPLE 1: 2×2 Matrix Inverse')
print('=' * 70)
print()

# Define 2×2 matrix
A_2x2 = np.array([[4, 7],
                  [2, 6]], dtype=float)

print('Matrix A:')
print(A_2x2)
print()

# Compute inverse using Gaussian elimination
inv_A_2x2_ge = compute_matrix_inverse_gaussian_elimination(A_2x2)

print('A^-1 (computed using Gaussian Elimination):')
print(inv_A_2x2_ge)
print()

# Compare with NumPy
inv_A_2x2_np = np.linalg.inv(A_2x2)

print('A^-1 (computed using np.linalg.inv):')
print(inv_A_2x2_np)
print()

# Verify: A @ A^-1 should equal I
identity_check = A_2x2 @ inv_A_2x2_ge
print('Verification: A @ A^-1 =')
print(identity_check)
print()
print(f'✓ Is identity matrix? {np.allclose(identity_check, np.eye(2))}')
print(f'✓ Matches NumPy result? {np.allclose(inv_A_2x2_ge, inv_A_2x2_np)}')

EXAMPLE 1: 2×2 Matrix Inverse

Matrix A:
[[4. 7.]
 [2. 6.]]

A^-1 (computed using Gaussian Elimination):
[[ 0.6 -0.7]
 [-0.2  0.4]]

A^-1 (computed using np.linalg.inv):
[[ 0.6 -0.7]
 [-0.2  0.4]]

Verification: A @ A^-1 =
[[ 1.00000000e+00 -1.11022302e-16]
 [ 1.11022302e-16  1.00000000e+00]]

✓ Is identity matrix? True
✓ Matches NumPy result? True


## Step 4: Example 2 - 3×3 Matrix Inverse

For a 3×3 matrix:
$$B = \begin{bmatrix} 1 & 2 & 3 \\ 0 & 1 & 4 \\ 5 & 6 & 0 \end{bmatrix}$$

We solve **3 systems**:
1. $B \cdot \mathbf{x}_1 = \mathbf{e}_1 = \begin{bmatrix} 1 \\ 0 \\ 0 \end{bmatrix}$
2. $B \cdot \mathbf{x}_2 = \mathbf{e}_2 = \begin{bmatrix} 0 \\ 1 \\ 0 \end{bmatrix}$
3. $B \cdot \mathbf{x}_3 = \mathbf{e}_3 = \begin{bmatrix} 0 \\ 0 \\ 1 \end{bmatrix}$

In [25]:
print('=' * 70)
print('EXAMPLE 2: 3×3 Matrix Inverse')
print('=' * 70)
print()

# Define 3×3 matrix
B_3x3 = np.array([[1, 2, 3],
                  [0, 1, 4],
                  [5, 6, 0]], dtype=float)

print('Matrix B:')
print(B_3x3)
print()

# Compute inverse using Gaussian elimination
inv_B_3x3_ge = compute_matrix_inverse_gaussian_elimination(B_3x3)

print('B^-1 (computed using Gaussian Elimination):')
print(inv_B_3x3_ge)
print()

# Compare with NumPy
inv_B_3x3_np = np.linalg.inv(B_3x3)

print('B^-1 (computed using np.linalg.inv):')
print(inv_B_3x3_np)
print()

# Verify: B @ B^-1 should equal I
identity_check = B_3x3 @ inv_B_3x3_ge
print('Verification: B @ B^-1 =')
print(np.round(identity_check, 10))  # Round to reduce floating point noise
print()
print(f'✓ Is identity matrix? {np.allclose(identity_check, np.eye(3))}')
print(f'✓ Matches NumPy result? {np.allclose(inv_B_3x3_ge, inv_B_3x3_np)}')

EXAMPLE 2: 3×3 Matrix Inverse

Matrix B:
[[1. 2. 3.]
 [0. 1. 4.]
 [5. 6. 0.]]

B^-1 (computed using Gaussian Elimination):
[[-24.  18.   5.]
 [ 20. -15.  -4.]
 [ -5.   4.   1.]]

B^-1 (computed using np.linalg.inv):
[[-24.  18.   5.]
 [ 20. -15.  -4.]
 [ -5.   4.   1.]]

Verification: B @ B^-1 =
[[ 1. -0. -0.]
 [ 0.  1.  0.]
 [ 0. -0.  1.]]

✓ Is identity matrix? True
✓ Matches NumPy result? True


## Step 5: Detailed Step-by-Step for 2×2 Case

Let's show the detailed steps for finding the inverse of the 2×2 matrix by solving each system individually.

In [26]:
print('=' * 70)
print('DETAILED: 2×2 Matrix Inverse - Solving Each System')
print('=' * 70)
print()

A = np.array([[4, 7],
              [2, 6]], dtype=float)

print('Matrix A:')
print(A)
print()
print('We want to find A^-1 such that A @ A^-1 = I')
print()
print('This requires solving 2 systems:')
print()

# System 1: A @ x_1 = e_1
print('-' * 70)
print('SYSTEM 1: A @ x_1 = e_1')
print('-' * 70)
e_1 = np.array([1, 0], dtype=float)
print('e_1 =', e_1)
print()
print('Augmented matrix [A | e_1]:')
aug_1 = np.hstack([A, e_1.reshape(-1, 1)])
print(aug_1)
print()

x_1 = gaussian_elimination(A, e_1)
print('Solution x_1 (using Gaussian Elimination):')
print(x_1)
print()
print('Verification: A @ x_1 =')
print(A @ x_1)
print()

# System 2: A @ x_2 = e_2
print('-' * 70)
print('SYSTEM 2: A @ x_2 = e_2')
print('-' * 70)
e_2 = np.array([0, 1], dtype=float)
print('e_2 =', e_2)
print()
print('Augmented matrix [A | e_2]:')
aug_2 = np.hstack([A, e_2.reshape(-1, 1)])
print(aug_2)
print()

x_2 = gaussian_elimination(A, e_2)
print('Solution x_2 (using Gaussian Elimination):')
print(x_2)
print()
print('Verification: A @ x_2 =')
print(A @ x_2)
print()

# Combine solutions as columns of A^-1
print('-' * 70)
print('COMBINE SOLUTIONS INTO A^-1')
print('-' * 70)
inv_A = np.column_stack([x_1, x_2])
print('A^-1 = [x_1 | x_2] =')
print(inv_A)
print()
print('Final verification: A @ A^-1 =')
print(A @ inv_A)
print('✓ Equals identity matrix I')

DETAILED: 2×2 Matrix Inverse - Solving Each System

Matrix A:
[[4. 7.]
 [2. 6.]]

We want to find A^-1 such that A @ A^-1 = I

This requires solving 2 systems:

----------------------------------------------------------------------
SYSTEM 1: A @ x_1 = e_1
----------------------------------------------------------------------
e_1 = [1. 0.]

Augmented matrix [A | e_1]:
[[4. 7. 1.]
 [2. 6. 0.]]

Solution x_1 (using Gaussian Elimination):
[ 0.6 -0.2]

Verification: A @ x_1 =
[1. 0.]

----------------------------------------------------------------------
SYSTEM 2: A @ x_2 = e_2
----------------------------------------------------------------------
e_2 = [0. 1.]

Augmented matrix [A | e_2]:
[[4. 7. 0.]
 [2. 6. 1.]]

Solution x_2 (using Gaussian Elimination):
[-0.7  0.4]

Verification: A @ x_2 =
[0. 1.]

----------------------------------------------------------------------
COMBINE SOLUTIONS INTO A^-1
----------------------------------------------------------------------
A^-1 = [x_1 | x_2] =


## Step 6: Detailed Step-by-Step for 3×3 Case (Column 1 Only)

Let's show the detailed steps for finding the first column of the 3×3 inverse.

In [27]:
print('=' * 70)
print('DETAILED: 3×3 Matrix Inverse - Finding Column 1 only')
print('=' * 70)
print()

B = np.array([[1, 2, 3],
              [0, 1, 4],
              [5, 6, 0]], dtype=float)

print('Matrix B:')
print(B)
print()
print('To find the first column of B^-1, we solve: B @ x_1 = e_1')
print()

e_1 = np.array([1, 0, 0], dtype=float)
print('e_1 =', e_1)
print()

print('Augmented matrix [B | e_1]:')
aug = np.hstack([B, e_1.reshape(-1, 1)])
print(aug)
print()

print('Solving using Gaussian Elimination...')
print()

x_1 = gaussian_elimination(B, e_1)
print('Solution x_1 (first column of B^-1):')
print(x_1)
print()

print('Verification: B @ x_1 =')
verify = B @ x_1
print(np.round(verify, 10))
print()
print('✓ Equals e_1')
print()
print('Similarly, we would solve B @ x_2 = e_2 and B @ x_3 = e_3')
print('to get the 2nd and 3rd columns of B^-1')

DETAILED: 3×3 Matrix Inverse - Finding Column 1 only

Matrix B:
[[1. 2. 3.]
 [0. 1. 4.]
 [5. 6. 0.]]

To find the first column of B^-1, we solve: B @ x_1 = e_1

e_1 = [1. 0. 0.]

Augmented matrix [B | e_1]:
[[1. 2. 3. 1.]
 [0. 1. 4. 0.]
 [5. 6. 0. 0.]]

Solving using Gaussian Elimination...

Solution x_1 (first column of B^-1):
[-24.  20.  -5.]

Verification: B @ x_1 =
[1. 0. 0.]

✓ Equals e_1

Similarly, we would solve B @ x_2 = e_2 and B @ x_3 = e_3
to get the 2nd and 3rd columns of B^-1


## Step 7: Comparison - Time Complexity

### Why Use Gaussian Elimination for Learning?

**Educational Value:**
- Understand the **mathematical principle**: matrix inverse = solving multiple linear systems
- See how each column of the inverse corresponds to a solution vector
- Learn the **row reduction** process that underpins linear algebra

**Computational Perspective:**
- **Gaussian Elimination Approach**: Solve $n$ systems, each $O(n^3)$ = $O(n^4)$ total
- **Direct Inversion** (np.linalg.inv): $O(n^3)$ using optimized algorithms
- For large $n$, direct inversion is **much faster**

**When to Use Each:**
| Method | When to Use | Advantage |
|--------|-----------|----------|
| Gaussian Elimination | Learning, debugging, small matrices | Understand the math |
| np.linalg.inv | Production code, large matrices | Fast, robust, optimized |
| np.linalg.solve | Solving Ax=b without finding A^-1 | Most efficient for linear systems |

In [28]:
import time

print('=' * 70)
print('PERFORMANCE COMPARISON')
print('=' * 70)
print()

# Define header texts
h1 = 'Matrix Size'
h2 = 'Gaussian Elim (s)'
h3 = 'NumPy Inv (s)'
h4 = 'Speedup (x)'
sep = ' | '

# Column widths derived from header text lengths
w1 = len(h1)
w2 = len(h2)
w3 = len(h3)
w4 = len(h4)

header = f'{h1:>{w1}}{sep}{h2:>{w2}}{sep}{h3:>{w3}}{sep}{h4:>{w4}}'
print(header)
print('-' * len(header))

for n in [2, 3, 4, 5, 6]:
    # Create random invertible matrix
    A = np.random.randn(n, n) + n * np.eye(n)  # Ensure invertibility
    
    # Time Gaussian Elimination
    start = time.time()
    A_inv_ge = compute_matrix_inverse_gaussian_elimination(A)
    time_ge = time.time() - start
    
    # Time NumPy
    start = time.time()
    A_inv_np = np.linalg.inv(A)
    time_np = time.time() - start
    
    speedup = time_ge / time_np if time_np > 0 else float('inf')
    row = f'{n:>{w1}d}{sep}{time_ge:>{w2}.6f}{sep}{time_np:>{w3}.6f}{sep}{speedup:>{w4}.1f}'
    print(row)

print()
print('Note: Times are very small; use for relative comparison only.')
print('For n > 10, numpy is typically 100-1000x faster!')

PERFORMANCE COMPARISON

Matrix Size | Gaussian Elim (s) | NumPy Inv (s) | Speedup (x)
-------------------------------------------------------------
          2 |          0.000149 |      0.000046 |         3.3
          3 |          0.000443 |      0.000039 |        11.5
          4 |          0.000501 |      0.000021 |        23.4
          5 |          0.000710 |      0.000018 |        39.2
          6 |          0.001355 |      0.000033 |        41.2

Note: Times are very small; use for relative comparison only.
For n > 10, numpy is typically 100-1000x faster!


## Summary

### Key Concepts

1. **Matrix Inverse via Linear Systems**
   - To find $A^{-1}$, solve: $A \cdot X = I$
   - This breaks down into $n$ separate systems: $A \cdot \mathbf{x}_i = \mathbf{e}_i$
   - Each solution $\mathbf{x}_i$ becomes column $i$ of $A^{-1}$

2. **Using Gaussian Elimination**
   - Forward elimination: Create zeros below pivots
   - Normalize pivot to 1
   - Backward elimination: Create zeros above pivots
   - Result: Reduced row echelon form with solution

3. **For 2×2 and 3×3 Matrices**
   - 2×2: Solve 2 systems of 2 equations each
   - 3×3: Solve 3 systems of 3 equations each
   - Each system yields one column of the inverse

4. **When to Use This Approach**
   - ✓ Educational: Understand matrix inverses and Gaussian elimination
   - ✓ Debugging: Verify small results by hand
   - ✗ Production: Use `np.linalg.inv()` (faster, more stable)
   - ✗ Solving Ax=b: Use `np.linalg.solve()` (don't need the inverse)